# Step 50 — Build ObsPy catalogs

One catalog builder supports two hypocenter association policies:

- `one_to_one` (default): greedy matching by smallest absolute time difference; each hypocenter and spine event is used at most once. This is not a global optimal assignment.
- `nearest`: original nearest-time behavior; the same hypocenter may be attached to multiple spine events.

Both use `origin_time_tol` seconds. Step-32 events, including pick-only events, form the authoritative spine. Unmatched hypocenters become H_ONLY events. Low-gain channel IDs are preserved from the pick index.

Review DATA_TOP and SETTINGS before running. Both policies use the same catalog assembly and export code. The canonical output prefix remains `50_pin_catalog`, so downstream steps 51/52 need no path changes. Set a separate prefix to compare policies without replacing canonical products. The former alternate notebook is retained under superseded/step50.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / 'pipeline', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths
from pinatubo_steps.step_50_build_obspy_catalog import run

DATA_TOP = Path.home() / 'Dropbox/PROFESSIONAL/DATA/Pinatubo/ASSEMBLED-25-028-THOMPSON-PINATUBO1991'
P = pipeline_paths(DATA_TOP)

SETTINGS = dict(
    waveform_event_index=P['WAVEFORM_PICK_EVENT_INDEX'],
    waveform_pick_map=P['PICK_MAP_CSV'],
    pick_index=P['MERGED_PICK_INDEX'],
    hypo_event_index=P['HYPO71_EVENT_INDEX'],
    hypo_origin_index=P['HYPO71_ORIGIN_INDEX'],
    origin_time_tol=float(P['ORIGIN_TIME_TOL_S']),
    out_prefix=P['CATALOG_PREFIX'],
    default_net='XB',
    association_policy='one_to_one',  # Or 'nearest' for the original policy.
)
display(SETTINGS)


## Run the step

Imports and settings above do not run the analysis. This cell writes the configured products.


In [ ]:
result = run(**SETTINGS)


In [ ]:
display(result['association_policy'])
display(result['class_counts'])
display(result['matches'].head(20))
print('All events:', len(result['catalog_all']))
print('Waveform-related events:', len(result['catalog_waveform']))
display(result['output_paths'])
